In [1]:
!pip install librosa transformers -q

import os, glob, pickle, random, json, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

PyTorch: 2.9.0+cu126
CUDA: True
Device: cuda


In [2]:
# ============================================================
# Set All Paths Directly
# ============================================================
from collections import defaultdict

# ✅ Set your exact paths here
RAW             = '/kaggle/input/datasets/mathurinache/cmu-mosi/Raw'
TRANSCRIPT_DIR  = f'{RAW}/Transcript/Segmented'
AUDIO_DIR       = f'{RAW}/Audio/WAV_16000/Segmented'
VIDEO_DIR       = f'{RAW}/Video/Segmented'

# Collect transcript .annotprocessed files
transcript_files = sorted([
    os.path.join(TRANSCRIPT_DIR, f)
    for f in os.listdir(TRANSCRIPT_DIR)
    if f.endswith('.annotprocessed')
])

# Collect audio .wav files
audio_wav_files = sorted([
    os.path.join(AUDIO_DIR, f)
    for f in os.listdir(AUDIO_DIR)
    if f.endswith('.wav')
])

# Collect video files (.mp4 or .avi)
video_files = sorted([
    os.path.join(VIDEO_DIR, f)
    for f in os.listdir(VIDEO_DIR)
    if f.endswith('.mp4') or f.endswith('.avi')
])

print(f"Transcript files : {len(transcript_files)}")
print(f"Audio .wav files : {len(audio_wav_files)}")
print(f"Video files      : {len(video_files)}")

print("\nSample transcript:", os.path.basename(transcript_files[0]))
print("Sample audio     :", os.path.basename(audio_wav_files[0]) if audio_wav_files else "none")
print("Sample video     :", os.path.basename(video_files[0]) if video_files else "none")

Transcript files : 93
Audio .wav files : 2199
Video files      : 2199

Sample transcript: 03bSnISJMiM.annotprocessed
Sample audio     : 03bSnISJMiM_1.wav
Sample video     : 03bSnISJMiM_1.mp4


In [3]:
# ============================================================
# Extract Acoustic Features using openSMILE
#           (COVAREP-equivalent: prosodic + voicing features)
# ============================================================
!pip install opensmile -q

import opensmile
from tqdm.notebook import tqdm

# ComParE_2016 gives 65 LLD functionals — we project to 74 to match paper dim
# eGeMAPSv02 gives 88 features — we truncate/pad to 74

smile = opensmile.Smile(
    feature_set=opensmile.FeatureSet.eGeMAPSv02,
    feature_level=opensmile.FeatureLevel.Functionals,
)

ACOUSTIC_DIM = 74  # keep same as before — matches paper

def extract_acoustic_opensmile(wav_path):
    try:
        features = smile.process_file(wav_path)
        feat = features.values[0].astype(np.float32)  # shape: (88,)
        # Truncate to 74 to match paper's COVAREP dim
        if len(feat) >= ACOUSTIC_DIM:
            return feat[:ACOUSTIC_DIM]
        # Pad if shorter (shouldn't happen with eGeMAPSv02)
        return np.pad(feat, (0, ACOUSTIC_DIM - len(feat))).astype(np.float32)
    except Exception as e:
        return np.zeros(ACOUSTIC_DIM, dtype=np.float32)

# Extract for all wav files
audio_data = {}
for wav_path in tqdm(audio_wav_files, desc="Extracting audio (openSMILE)"):
    video_id = os.path.splitext(os.path.basename(wav_path))[0]
    feat     = extract_acoustic_opensmile(wav_path)
    audio_data[video_id] = {
        'features': feat,
        'video_id': video_id.rsplit('_', 1)[0]
    }

print(f"Audio features extracted : {len(audio_data)}")
print(f"Acoustic dim             : {ACOUSTIC_DIM}")
print(f"Sample feature shape     : {audio_data[list(audio_data.keys())[0]]['features'].shape}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.7/133.7 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.0/325.0 kB 18.0 MB/s eta 0:00:00


Extracting audio (openSMILE):   0%|          | 0/2199 [00:00<?, ?it/s]

Audio features extracted : 2199
Acoustic dim             : 74
Sample feature shape     : (74,)


In [4]:
# ============================================================
# Extract Visual Features from Video Files
# ============================================================
import cv2
import torchvision.models as tvm
import torchvision.transforms as T


VISUAL_DIM = 35  # matches paper (we'll PCA/project down)

# Load ResNet and remove final layer
resnet  = tvm.resnet18(pretrained=True)
resnet  = nn.Sequential(*list(resnet.children())[:-1]).to(device)
resnet.eval()

# Project 512 → 35
vis_proj = nn.Linear(512, VISUAL_DIM).to(device)
nn.init.xavier_uniform_(vis_proj.weight)

transform = T.Compose([
    T.ToPILImage(),
    T.Resize(224), T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

def extract_visual(video_path, n_frames=5):
    try:
        cap   = cv2.VideoCapture(video_path)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total == 0: return np.zeros(VISUAL_DIM, dtype=np.float32)
        idxs  = np.linspace(0, total-1, n_frames, dtype=int)
        frames = []
        for idx in idxs:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if ret:
                frames.append(transform(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
        cap.release()
        if not frames: return np.zeros(VISUAL_DIM, dtype=np.float32)
        with torch.no_grad():
            batch = torch.stack(frames).to(device)
            feat  = resnet(batch).squeeze(-1).squeeze(-1).mean(0, keepdim=True)
            feat  = vis_proj(feat).squeeze(0)
        return feat.cpu().numpy().astype(np.float32)
    except:
        return np.zeros(VISUAL_DIM, dtype=np.float32)

video_data = {}
for vid_path in tqdm(video_files, desc="Extracting video"):
    video_id = os.path.splitext(os.path.basename(vid_path))[0]
    feat     = extract_visual(vid_path)
    video_data[video_id] = {'features': feat, 'video_id': video_id.rsplit('_',1)[0]}

print(f"Video features extracted: {len(video_data)}")
print(f"Visual dim: {VISUAL_DIM}")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 224MB/s]


Extracting video:   0%|          | 0/2199 [00:00<?, ?it/s]

Video features extracted: 2199
Visual dim: 35


In [5]:
# ============================================================
# Parse Transcript Files + Extract Text
# ============================================================
# Transcript format: "1_DELIM_ UTTERANCE TEXT"
# Audio files named: "03bSnISJMiM_1.wav"  → segment_id = "03bSnISJMiM_1"
# We need to match transcript segment index to audio/video file names
def parse_transcript_file(filepath):
    """
    Parse Transcript .annotprocessed file.
    Returns list of dicts: {video_id, segment_id, text}
    segment_id matches audio/video filenames e.g. "03bSnISJMiM_1"
    """
    video_id = os.path.splitext(os.path.basename(filepath))[0]
    results  = []
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        lines = f.readlines()
    for line in lines:
        line = line.strip()
        if not line:
            continue
        if '_DELIM_' in line:
            parts   = line.split('_DELIM_', 1)
            seg_idx = parts[0].strip()       # "1", "2", "3" ...
            text    = parts[1].strip()
        else:
            seg_idx = str(len(results) + 1)
            text    = line
        results.append({
            'video_id':   video_id,
            'segment_id': f"{video_id}_{seg_idx}",
            # "03bSnISJMiM_1"
            'text':       text,
        })
    return results


# Load all transcripts
transcript_segments = {}   # {segment_id: {video_id, text}}
for fpath in transcript_files:
    for item in parse_transcript_file(fpath):
        transcript_segments[item['segment_id']] = item

print(f"Total transcript segments : {len(transcript_segments)}")
print(f"Audio segments loaded     : {len(audio_data)}")
print(f"Video segments loaded     : {len(video_data)}")

# Verify key matching
sample_key = list(transcript_segments.keys())[0]
print(f"\nSample transcript key : '{sample_key}'")
print(f"In audio_data         : {sample_key in audio_data}")
print(f"In video_data         : {sample_key in video_data}")



Total transcript segments : 2199
Audio segments loaded     : 2199
Video segments loaded     : 2199

Sample transcript key : '03bSnISJMiM_1'
In audio_data         : True
In video_data         : True


In [6]:
# ============================================================
# Extract Text Features with BERT
# ============================================================
from transformers import AutoTokenizer, AutoModel
from tqdm.notebook import tqdm

print("Loading BERT (bert-base-uncased)...")
tokenizer  = AutoTokenizer.from_pretrained('bert-base-uncased')
bert_model = AutoModel.from_pretrained('bert-base-uncased').to(device)
bert_model.eval()
TEXT_DIM = 768

seg_ids     = list(transcript_segments.keys())
text_features = {}   # {segment_id: np.array (768,)}

BATCH = 64
for i in tqdm(range(0, len(seg_ids), BATCH), desc="BERT text features"):
    batch_ids   = seg_ids[i : i + BATCH]
    batch_texts = [transcript_segments[s]['text'] for s in batch_ids]
    inputs = tokenizer(
        batch_texts,
        return_tensors='pt',
        truncation=True,
        max_length=64,
        padding=True
    ).to(device)
    with torch.no_grad():
        out = bert_model(**inputs)
        cls = out.last_hidden_state[:, 0, :]   # (B, 768)
    for j, sid in enumerate(batch_ids):
        text_features[sid] = cls[j].cpu().numpy().astype(np.float32)

print(f"✓ Text features extracted : {len(text_features)}")
print(f"  TEXT_DIM = {TEXT_DIM}")


Loading BERT (bert-base-uncased)...


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT text features:   0%|          | 0/35 [00:00<?, ?it/s]

✓ Text features extracted : 2199
  TEXT_DIM = 768


In [7]:
# ============================================================
# SAVE MULTIMODAL FEATURES TO CMU-MOSI FORMATTED PICKLE FILE
# ============================================================
import os
import pickle
import numpy as np

# Set destination path (matches the target MOSI structure)
SAVE_DIR = "/kaggle/working/processed_dataset"
os.makedirs(SAVE_DIR, exist_ok=True)
PICKLE_OUTPUT_PATH = os.path.join(SAVE_DIR, "mosi_data.pkl")

# Collect all valid segment IDs present across modalities
valid_segment_ids = sorted(
    list(
        set(transcript_segments.keys())
        & set(audio_data.keys())
        & set(video_data.keys())
    )
)

print(f"Total synchronized segment IDs ready for export: {len(valid_segment_ids)}")

# Group extracted features by video_id to retain hierarchical sequence structure
video_to_segments = {}
for seg_id in valid_segment_ids:
    vid_id = transcript_segments[seg_id]['video_id']
    if vid_id not in video_to_segments:
        video_to_segments[vid_id] = []
    video_to_segments[vid_id].append(seg_id)

# Helper function to sort segment IDs by numerical order (e.g., video_1, video_2, ..., video_10)
def get_seg_num(seg_id):
    try:
        return int(seg_id.rsplit('_', 1)[1])
    except ValueError:
        return 0

# Construct sequences per video
dataset_dict = {
    'text': {},
    'audio': {},
    'vision': {},
    'labels': {},
    'transcripts': {}
}

for vid_id, seg_list in video_to_segments.items():
    # Sort segments sequentially
    seg_list = sorted(seg_list, key=get_seg_num)
    
    # Sequence matrices: shape -> (num_utterances, feature_dim)
    t_seq = np.stack([text_features[sid] for sid in seg_list], axis=0)
    a_seq = np.stack([audio_data[sid]['features'] for sid in seg_list], axis=0)
    v_seq = np.stack([video_data[sid]['features'] for sid in seg_list], axis=0)
    
    # Store raw transcripts for downstream debugging/analysis
    raw_texts = [transcript_segments[sid]['text'] for sid in seg_list]
    
    # Dummy/Placeholder sentiment labels (Replace with ground-truth sentiment dict if available)
    # Shape: (num_utterances,) or (num_utterances, 1)
    labels = np.zeros((len(seg_list), 1), dtype=np.float32)

    dataset_dict['text'][vid_id] = t_seq
    dataset_dict['audio'][vid_id] = a_seq
    dataset_dict['vision'][vid_id] = v_seq
    dataset_dict['labels'][vid_id] = labels
    dataset_dict['transcripts'][vid_id] = raw_texts

# Save payload to pickle file
with open(PICKLE_OUTPUT_PATH, "wb") as f:
    pickle.dump(dataset_dict, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"\n✓ Dataset successfully compiled and saved to: {PICKLE_OUTPUT_PATH}")
print(f"Total Videos Processed: {len(dataset_dict['text'])}")

# Verification step
with open(PICKLE_OUTPUT_PATH, "rb") as f:
    loaded_data = pickle.load(f)

sample_vid = list(loaded_data['text'].keys())[0]
print("\n--- Pickle Verification ---")
print(f"Sample Video ID : {sample_vid}")
print(f"Text shape     : {loaded_data['text'][sample_vid].shape}")      # (num_utt, 768)
print(f"Audio shape    : {loaded_data['audio'][sample_vid].shape}")     # (num_utt, 74)
print(f"Vision shape   : {loaded_data['vision'][sample_vid].shape}")    # (num_utt, 35)
print(f"Labels shape   : {loaded_data['labels'][sample_vid].shape}")    # (num_utt, 1)

Total synchronized segment IDs ready for export: 2199

✓ Dataset successfully compiled and saved to: /kaggle/working/processed_dataset/mosi_data.pkl
Total Videos Processed: 93

--- Pickle Verification ---
Sample Video ID : 03bSnISJMiM
Text shape     : (13, 768)
Audio shape    : (13, 74)
Vision shape   : (13, 35)
Labels shape   : (13, 1)


In [8]:
# # ============================================================
# # CELL — PCA: BERT 768 → 100
# # ============================================================
# from sklearn.decomposition import PCA

# # Use text_features keys directly (guaranteed to exist)
# seg_ids_sorted    = sorted(text_features.keys())
# all_feats         = np.stack([text_features[s] for s in seg_ids_sorted])

# print(f"Matrix shape: {all_feats.shape}")   # should be (2199, 768)

# del bert_model
# torch.cuda.empty_cache()

# pca               = PCA(n_components=100, random_state=42)
# all_feats_reduced = pca.fit_transform(all_feats)

# for i, sid in enumerate(seg_ids_sorted):
#     text_features[sid] = all_feats_reduced[i].astype(np.float32)

# TEXT_DIM = 100
# print(f"✓ TEXT_DIM = {TEXT_DIM}  |  Variance kept: {pca.explained_variance_ratio_.sum()*100:.1f}%")

In [9]:
# # ============================================================
# # CELL — Load saved PCA text features
# # ============================================================
# import numpy as np

# all_feats_reduced = np.load('/kaggle/working/text_feats_pca100.npy')
# seg_ids_sorted    = np.load('/kaggle/working/seg_ids_sorted.npy', allow_pickle=True).tolist()

# text_features = {}
# for i, sid in enumerate(seg_ids_sorted):
#     text_features[sid] = all_feats_reduced[i].astype(np.float32)

# TEXT_DIM = 100
# print(f"✓ Loaded {len(text_features)} text features  |  TEXT_DIM={TEXT_DIM}")

In [10]:
import pickle
import pandas as pd

path = "/kaggle/input/datasets/reganwillis/cmu-mosi/mosi_data.pkl"

with open(path, "rb") as f:
    data = pickle.load(f)

rows = []

for split in ["train", "valid", "test"]:
    
    ids = data[split]["id"]
    labels = data[split]["labels"]
    
    for i in range(len(ids)):
        
        segment_id = ids[i][0].decode("utf-8")
        start_time = float(ids[i][1])
        end_time = float(ids[i][2])
        sentiment = float(labels[i][0])
        
        rows.append({
            "segment_id": segment_id,
            "start": start_time,
            "end": end_time,
            "sentiment": sentiment
        })

df = pd.DataFrame(rows)

df.head()

,segment_id,start,end,sentiment
0,03bSnISJMiM_11,95.056463,97.201587,-0.5
1,03bSnISJMiM_10,93.549887,95.056463,-1.2
2,03bSnISJMiM_13,168.838775,170.485034,1.8
3,03bSnISJMiM_12,97.201587,99.895465,2.2
4,03bSnISJMiM_1,52.732653,56.045125,2.4


In [11]:
print(data.keys()) 
print(data['train'].keys())

print(type(data['train']['id']))
print(type(data['train']['audio']))

print(data['train']['audio'].shape)
print(data['train']['vision'].shape)

print(data['train']['id'][0])
print(data['train']['labels'][0])

dict_keys(['valid', 'test', 'train'])
dict_keys(['vision', 'labels', 'text', 'audio', 'id'])
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
(1284, 50, 5)
(1284, 50, 20)
[b'03bSnISJMiM_11' b'95.05646259' b'97.2015873']
[[-0.5]]


In [12]:
print(data['train']['text'].shape)
print(data['train']['vision'].shape)
print(data['train']['audio'].shape)

(1284, 50, 300)
(1284, 50, 20)
(1284, 50, 5)


In [13]:
# Add split column to DataFrame if not present
df_split = []
for split in ["train", "valid", "test"]:
    for i in range(len(data[split]["id"])):
        df_split.append({
            "split": split,
            "segment_id": data[split]["id"][i][0].decode("utf-8"),
            "sentiment": float(data[split]["labels"][i][0])
        })
df_stats = pd.DataFrame(df_split)

# Group by split and analyze
print(df_stats.groupby('split')['sentiment'].describe())

        count      mean       std  min  25%  50%  75%  max
split                                                     
test    686.0 -0.316983  1.589544 -3.0 -1.6 -0.6  1.0  3.0
train  1284.0  0.234489  1.510039 -3.0 -1.0  0.2  1.6  3.0
valid   229.0  0.309534  1.652841 -3.0 -1.2  0.4  1.8  3.0


In [14]:
df.describe()

,start,end,sentiment
count,2199.000000,2199.000000,2199.000000
mean,85.966739,90.195614,0.070267
std,48.273726,48.383443,1.571724
min,0.052381,1.778458,-3.000000
25%,46.207483,50.063719,-1.250000
50%,82.744444,87.453742,0.000000
75%,125.252834,129.528118,1.400000
max,179.504535,180.043311,3.000000


In [15]:
# ============================================================
# CELL — Fix Segment ID Mapping
# ============================================================
# Your labels have:   "03bSnISJMiM_11"  (video_id + "_" + seg_number)
# Your audio files:   "03bSnISJMiM_1.wav"
# Your transcripts:   "1_DELIM_ TEXT"  → parsed as "03bSnISJMiM_1"
# These should all match — let's verify

# Check overlap
label_keys      = set(df['segment_id'])
transcript_keys = set(transcript_segments.keys())
audio_keys      = set(audio_data.keys())
video_keys      = set(video_data.keys())

print(f"Label keys      : {len(label_keys)}")
print(f"Transcript keys : {len(transcript_keys)}")
print(f"Audio keys      : {len(audio_keys)}")
print(f"Video keys      : {len(video_keys)}")

print(f"\nLabel ∩ Transcript : {len(label_keys & transcript_keys)}")
print(f"Label ∩ Audio      : {len(label_keys & audio_keys)}")
print(f"Label ∩ Video      : {len(label_keys & video_keys)}")

# Print sample keys from each to see format
print(f"\nSample label key      : {sorted(label_keys)[:3]}")
print(f"Sample transcript key : {sorted(transcript_keys)[:3]}")
print(f"Sample audio key      : {sorted(audio_keys)[:3]}")
print(f"Sample video key      : {sorted(video_keys)[:3]}")

Label keys      : 2199
Transcript keys : 2199
Audio keys      : 2199
Video keys      : 2199

Label ∩ Transcript : 2199
Label ∩ Audio      : 2199
Label ∩ Video      : 2199

Sample label key      : ['03bSnISJMiM_1', '03bSnISJMiM_10', '03bSnISJMiM_11']
Sample transcript key : ['03bSnISJMiM_1', '03bSnISJMiM_10', '03bSnISJMiM_11']
Sample audio key      : ['03bSnISJMiM_1', '03bSnISJMiM_10', '03bSnISJMiM_11']
Sample video key      : ['03bSnISJMiM_1', '03bSnISJMiM_10', '03bSnISJMiM_11']


In [16]:
# ============================================================
# CELL — Build label_data dict from df
# ============================================================
label_data = dict(zip(df['segment_id'], df['sentiment']))

print(f"Labels loaded : {len(label_data)}")
print(f"Score range   : {min(label_data.values()):.2f} to {max(label_data.values()):.2f}")
print(f"Positive (>=0): {sum(1 for v in label_data.values() if v >= 0)}")
print(f"Negative (<0) : {sum(1 for v in label_data.values() if v < 0)}")

Labels loaded : 2199
Score range   : -3.00 to 3.00
Positive (>=0): 1176
Negative (<0) : 1023


In [17]:
# ============================================================
# Build Aligned Dataset (all 2199 match, use all)
# ============================================================
def safe_feat(d, seg_id, dim):
    if seg_id not in d: return np.zeros(dim, dtype=np.float32)
    f = d[seg_id]['features']
    if len(f) >= dim: return f[:dim].astype(np.float32)
    return np.pad(f, (0, dim - len(f))).astype(np.float32)

samples = []
for seg_id in sorted(label_data.keys()):
    score = label_data[seg_id]
    samples.append({
        'text':     text_features[seg_id].reshape(1, -1),
        'visual':   safe_feat(video_data, seg_id, VISUAL_DIM).reshape(1, -1),
        'acoustic': safe_feat(audio_data, seg_id, ACOUSTIC_DIM).reshape(1, -1),
        'label':    1 if score >= 0 else 0,
        'score':    score,
        'video_id': seg_id.rsplit('_', 1)[0],
    })

print(f"Total samples  : {len(samples)}")
print(f"Positive (>=0) : {sum(1 for s in samples if s['label']==1)}")
print(f"Negative (<0)  : {sum(1 for s in samples if s['label']==0)}")
print(f"Text dim       : {TEXT_DIM}")
print(f"Visual dim     : {VISUAL_DIM}")
print(f"Acoustic dim   : {ACOUSTIC_DIM}")

Total samples  : 2199
Positive (>=0) : 1176
Negative (<0)  : 1023
Text dim       : 768
Visual dim     : 35
Acoustic dim   : 74


In [18]:
from collections import defaultdict

# Group utterances by video, sorted by index
video_utts = defaultdict(list)
for seg_id in sorted(label_data.keys()):
    vid_id = seg_id.rsplit('_', 1)[0]
    video_utts[vid_id].append({
        'seg_id':   seg_id,
        'text':     text_features[seg_id],
        'visual':   safe_feat(video_data,  seg_id, VISUAL_DIM),
        'acoustic': safe_feat(audio_data,  seg_id, ACOUSTIC_DIM),
        'label':    1 if label_data[seg_id] >= 0 else 0,
    })

def seg_sort_key(u):
    try: return int(u['seg_id'].rsplit('_', 1)[1])
    except: return 0

for vid in video_utts:
    video_utts[vid].sort(key=seg_sort_key)

video_ids = sorted(video_utts.keys())
print(f"Total videos     : {len(video_ids)}")
print(f"Total utterances : {sum(len(v) for v in video_utts.values())}")
utts = [len(v) for v in video_utts.values()]
print(f"Utts/video       : min={min(utts)} max={max(utts)} mean={np.mean(utts):.1f}")



Total videos     : 93
Total utterances : 2199
Utts/video       : min=9 max=63 mean=23.6


In [19]:
# ============================================================
# CELL B — Split by VIDEO (paper: 52 train / 10 val / 31 test)
# ============================================================
random.seed(42)
shuffled = video_ids.copy()
random.shuffle(shuffled)
n    = len(shuffled)
n_tr = int(0.65 * n)   # ~60 videos
n_vl = int(0.10 * n)   # ~9 videos

train_vids = set(shuffled[:n_tr])
valid_vids = set(shuffled[n_tr : n_tr + n_vl])
test_vids  = set(shuffled[n_tr + n_vl:])

print(f"Train videos : {len(train_vids)}  "
      f"({sum(len(video_utts[v]) for v in train_vids)} utterances)")
print(f"Valid videos : {len(valid_vids)}  "
      f"({sum(len(video_utts[v]) for v in valid_vids)} utterances)")
print(f"Test  videos : {len(test_vids)}  "
      f"({sum(len(video_utts[v]) for v in test_vids)} utterances)")


Train videos : 60  (1412 utterances)
Valid videos : 9  (206 utterances)
Test  videos : 24  (581 utterances)


In [20]:
import collections, pandas as pd

# Count utterances per video ID across all dataset splits
video_counts = collections.Counter([v.decode('utf-8').rsplit('_', 1)[0] for split in ['train', 'valid', 'test'] for v in data[split]['id'][:, 0]])

# Display video IDs alongside their respective utterance counts
print(pd.Series(video_counts, name="Utterance Count"))

03bSnISJMiM    13
0h-zjBukYpk    25
1DmNV9C1hbY    14
1iG0909rllw    30
2WGyTLYerpo    63
               ..
vyB00TXsimI    22
wMbj6ajWbic    30
yDtzw_Y-7RU    24
yvsjCA6Y5Fc    23
zhpQhgha_KU    35
Name: Utterance Count, Length: 93, dtype: int64


In [21]:
# ============================================================
# CELL C — Dataset: one sample = one video, labels per utterance
# ============================================================
MAX_UTT = 63


def pad_seq(arr, max_len):
    arr = np.array(arr, dtype=np.float32)
    N, D = arr.shape
    if N >= max_len:
        return arr[:max_len], max_len
    pad = np.zeros((max_len - N, D), dtype=np.float32)
    return np.concatenate([arr, pad]), N

class VideoSeqDataset(Dataset):
    """
    Each item = one video.
    Returns padded sequences + per-utterance labels.
    """
    def __init__(self, vid_ids, video_utts, max_utt=MAX_UTT):
        self.vids     = list(vid_ids)
        self.utts     = video_utts
        self.max_utt  = max_utt

    def __len__(self): return len(self.vids)

    def __getitem__(self, idx):
        vid  = self.vids[idx]
        data = self.utts[vid]
        N    = min(len(data), self.max_utt)

        txt, _ = pad_seq(np.stack([u['text']     for u in data]), self.max_utt)
        vis, _ = pad_seq(np.stack([u['visual']   for u in data]), self.max_utt)
        aco, _ = pad_seq(np.stack([u['acoustic'] for u in data]), self.max_utt)

        # Per-utterance labels — pad with -1 (ignored in loss)
        labels = np.full(self.max_utt, -1, dtype=np.int64)
        for i, u in enumerate(data[:self.max_utt]):
            labels[i] = u['label']

        mask = np.zeros(self.max_utt, dtype=bool)
        mask[:N] = True

        return {
            'text':     torch.tensor(txt,    dtype=torch.float32),
            'visual':   torch.tensor(vis,    dtype=torch.float32),
            'acoustic': torch.tensor(aco,    dtype=torch.float32),
            'labels':   torch.tensor(labels, dtype=torch.long),
            'mask':     torch.tensor(mask,   dtype=torch.bool),
        }

BATCH = 8

train_ds = VideoSeqDataset(train_vids, video_utts, MAX_UTT)
valid_ds = VideoSeqDataset(valid_vids, video_utts, MAX_UTT)
test_ds  = VideoSeqDataset(test_vids,  video_utts, MAX_UTT)

train_loader = DataLoader(train_ds, BATCH, shuffle=True,  num_workers=2, pin_memory=True)
valid_loader = DataLoader(valid_ds, BATCH, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  BATCH, shuffle=False, num_workers=2, pin_memory=True)

b = next(iter(train_loader))
print(f"text     : {b['text'].shape}")      # (8, 63, 100)
print(f"visual   : {b['visual'].shape}")    # (8, 63, 35)
print(f"acoustic : {b['acoustic'].shape}")  # (8, 63, 74)
print(f"labels   : {b['labels'].shape}")    # (8, 63)  — per utterance

text     : torch.Size([8, 63, 768])
visual   : torch.Size([8, 63, 35])
acoustic : torch.Size([8, 63, 74])
labels   : torch.Size([8, 63])


In [22]:
# ============================================================
# Training Functions & MulT Architecture Updates
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import random
import numpy as np

def set_seed(s=42):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)

# ------------------------------------------------------------
# Enhanced MulT Architecture with Residual Text Bypass & LayerNorm
# ------------------------------------------------------------
class CrossmodalTransformerBlock(nn.Module):
    def __init__(self, embed_dim, nhead=2, dropout=0.3):
        super().__init__()
        self.mha = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=nhead, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * 2, embed_dim),
            nn.Dropout(dropout)
        )

    def forward(self, target_q, source_kv):
        attn_out, _ = self.mha(query=target_q, key=source_kv, value=source_kv)
        x = self.norm1(target_q + attn_out)
        out = self.norm2(x + self.ffn(x))
        return out

class MulT_PerUtterance(nn.Module):
    def __init__(self, td=768, vd=35, ad=74, d=64, nhead=2, num_classes=2, dropout=0.4):
        super().__init__()
        
        # Modality Projections with LayerNorm & Modality-Specific Dropout
        self.proj_t = nn.Sequential(nn.Linear(td, d), nn.LayerNorm(d), nn.Dropout(0.2))
        self.proj_v = nn.Sequential(nn.Linear(vd, d), nn.LayerNorm(d), nn.Dropout(0.5)) # Stronger visual regularization
        self.proj_a = nn.Sequential(nn.Linear(ad, d), nn.LayerNorm(d), nn.Dropout(0.5)) # Stronger acoustic regularization
        
        # Crossmodal Transformer Blocks
        self.cm_a2t = CrossmodalTransformerBlock(d, nhead, dropout)
        self.cm_v2t = CrossmodalTransformerBlock(d, nhead, dropout)
        
        self.cm_t2v = CrossmodalTransformerBlock(d, nhead, dropout)
        self.cm_a2v = CrossmodalTransformerBlock(d, nhead, dropout)
        
        self.cm_t2a = CrossmodalTransformerBlock(d, nhead, dropout)
        self.cm_v2a = CrossmodalTransformerBlock(d, nhead, dropout)
        
        # Self-Attention Encoders
        encoder_layer = nn.TransformerEncoderLayer(d_model=2*d, nhead=nhead, dim_feedforward=2*d, dropout=dropout, batch_first=True)
        self.self_attn_t = nn.TransformerEncoder(encoder_layer, num_layers=1)
        self.self_attn_v = nn.TransformerEncoder(encoder_layer, num_layers=1)
        self.self_attn_a = nn.TransformerEncoder(encoder_layer, num_layers=1)
        
        # Final Head includes direct Text Skip Connection (2d + 2d + 2d + d = 7d)
        self.fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(7 * d, d),
            nn.ReLU(),
            nn.Linear(d, num_classes)
        )

    def forward(self, t, v, a):
        T = self.proj_t(t)
        V = self.proj_v(v)
        A = self.proj_a(a)
        
        # Crossmodal Fusion
        t_fused = torch.cat([self.cm_a2t(T, A), self.cm_v2t(T, V)], dim=-1)
        v_fused = torch.cat([self.cm_t2v(V, T), self.cm_a2v(V, A)], dim=-1)
        a_fused = torch.cat([self.cm_t2a(A, T), self.cm_v2a(A, V)], dim=-1)
        
        # Self-Attention Processing
        t_out = self.self_attn_t(t_fused)
        v_out = self.self_attn_v(v_fused)
        a_out = self.self_attn_a(a_fused)
        
        # Concatenate Crossmodal Streams + Residual Text Stream
        out = torch.cat([t_out, v_out, a_out, T], dim=-1)
        
        logits = self.fc(out)
        return logits

# ------------------------------------------------------------
# Epoch Execution and Full Model Training Loops
# ------------------------------------------------------------
def run_epoch(model, loader, opt, crit, mods, train=True):
    model.train() if train else model.eval()
    total_loss, all_preds, all_golds = 0.0, [], []
    n_utts = 0

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for b in loader:
            t = b['text'].to(device)     if 'text'     in mods else torch.zeros_like(b['text']).to(device)
            v = b['visual'].to(device)   if 'visual'   in mods else torch.zeros_like(b['visual']).to(device)
            a = b['acoustic'].to(device) if 'acoustic' in mods else torch.zeros_like(b['acoustic']).to(device)
            y = b['labels'].to(device)   # (B, u) — -1 for padded utterances

            if train: 
                opt.zero_grad()

            logits = model(t, v, a)      # (B, u, 2)
            B, u, C = logits.shape

            logits_flat = logits.view(B * u, C)
            y_flat      = y.view(B * u)
            valid       = y_flat != -1   # Filter zero-padded utterances

            loss = crit(logits_flat[valid], y_flat[valid])

            if train:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()

            n_valid     = valid.sum().item()
            total_loss += loss.item() * n_valid
            n_utts      += n_valid

            preds = logits_flat[valid].argmax(1).cpu().numpy()
            golds = y_flat[valid].cpu().numpy()
            all_preds.extend(preds)
            all_golds.extend(golds)

    acc = accuracy_score(all_golds, all_preds) * 100
    return total_loss / n_utts, acc, all_preds, all_golds


def train_mult_model(mods=['text', 'visual', 'acoustic'], seed=42, verbose=True):
    set_seed(seed)
    
    model = MulT_PerUtterance(
        td=TEXT_DIM, vd=VISUAL_DIM, ad=ACOUSTIC_DIM,
        d=PROJ_DIM, nhead=NUM_HEADS, num_classes=NUM_CLASSES, dropout=DROPOUT
    ).to(device)
    
    opt = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-6)
    crit = nn.CrossEntropyLoss()
    
    best_val_loss = float('inf')
    best_state = None
    patience_counter = 0
    hist = {'tl': [], 'ta': [], 'vl': [], 'va': []}

    for ep in range(1, EPOCHS + 1):
        tl, ta, _, _ = run_epoch(model, train_loader, opt, crit, mods, train=True)
        vl, va, _, _ = run_epoch(model, valid_loader, opt, crit, mods, train=False)
        
        scheduler.step()
        
        hist['tl'].append(tl); hist['ta'].append(ta)
        hist['vl'].append(vl); hist['va'].append(va)
        
        # Save model based on minimum validation loss
        if vl < best_val_loss:
            best_val_loss = vl
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            
        if verbose and ep % 5 == 0:
            print(f"  Ep {ep:3d} | Train Loss: {tl:.4f} Acc: {ta:.1f}% | Val Loss: {vl:.4f} Acc: {va:.1f}%")

        if patience_counter >= PATIENCE:
            print(f"\n! Early stopping triggered at Epoch {ep} (Best Val Loss: {best_val_loss:.4f})")
            break

    # Load best checkpoint and evaluate on unseen test data
    model.load_state_dict(best_state)
    _, test_acc, preds, golds = run_epoch(model, test_loader, opt, crit, mods, train=False)
    
    return test_acc, preds, golds, model, hist

print("✓ Updated MulT CELL F ready")

✓ Updated MulT CELL F ready


In [23]:
# ============================================================
# CELL E — Hyperparameters (Optimized for MulT)
# ============================================================
# PROJ_DIM    = 100    # Unified embedding dimension d across modalities
# NUM_HEADS   = 4      # Attention heads for MultiheadAttention
# DROPOUT     = 0.3    # Attention & Linear dropout rate
# LR          = 3e-4   # Base learning rate for Transformers
# WEIGHT_DECAY= 1e-3   # L2 regularization
# EPOCHS      = 50
# NUM_CLASSES = 2


# PROJ_DIM     = 64     # Reduce projection dim from 100 -> 64 to limit capacity
# NUM_HEADS    = 2      # Reduce heads from 4 -> 2
# DROPOUT      = 0.4    # Stronger dropout
# LR           = 1e-4   # Slightly lower base learning rate
# WEIGHT_DECAY = 5e-3   # Stronger weight decay
# PATIENCE     = 8      # Early stopping patience


# ============================================================
# CELL E — Optimized Hyperparameters for MulT Stability
# ============================================================
PROJ_DIM     = 64      # Lower capacity to prevent noise memorization
NUM_HEADS    = 2       # 2 heads work best for smaller sequence dimensions
DROPOUT      = 0.4     # Crossmodal attention dropout
LR           = 1e-4    # Lower learning rate for stable transformer convergence
WEIGHT_DECAY = 1e-2    # L2 regularization
PATIENCE     = 10      # Early stopping window
EPOCHS       = 50
NUM_CLASSES  = 2

print(f"TEXT:{TEXT_DIM} | VISUAL:{VISUAL_DIM} | ACOUSTIC:{ACOUSTIC_DIM}")

print(f"TEXT:{TEXT_DIM} | VISUAL:{VISUAL_DIM} | ACOUSTIC:{ACOUSTIC_DIM}")
print(f"MulT Setup -> Projection Dim: {PROJ_DIM} | Attention Heads: {NUM_HEADS} | LR: {LR}")

TEXT:768 | VISUAL:35 | ACOUSTIC:74
TEXT:768 | VISUAL:35 | ACOUSTIC:74
MulT Setup -> Projection Dim: 64 | Attention Heads: 2 | LR: 0.0001


In [24]:
# ============================================================
# CELL G — Train Multimodal Transformer (MulT)
# ============================================================
print("=" * 60)
print("MulT Architecture — Tri-modal | Per-utterance prediction")
print("=" * 60)

acc, preds, golds, mult_model, hist = train_mult_model(
    mods=['text', 'visual', 'acoustic'], seed=42, verbose=True
)

print(f"\n✓ Test Accuracy : {acc:.2f}%")
print(classification_report(golds, preds, target_names=['Negative', 'Positive'], digits=3))
print("Confusion Matrix:\n", confusion_matrix(golds, preds))

MulT Architecture — Tri-modal | Per-utterance prediction
  Ep   5 | Train Loss: 0.6470 Acc: 63.5% | Val Loss: 0.6455 Acc: 64.1%
  Ep  10 | Train Loss: 0.5347 Acc: 75.3% | Val Loss: 0.5561 Acc: 74.8%
  Ep  15 | Train Loss: 0.4567 Acc: 78.8% | Val Loss: 0.5272 Acc: 75.7%
  Ep  20 | Train Loss: 0.4023 Acc: 82.3% | Val Loss: 0.5255 Acc: 76.7%
  Ep  25 | Train Loss: 0.3869 Acc: 83.9% | Val Loss: 0.5254 Acc: 77.2%

! Early stopping triggered at Epoch 29 (Best Val Loss: 0.5182)

✓ Test Accuracy : 79.52%
              precision    recall  f1-score   support

    Negative      0.750     0.794     0.772       253
    Positive      0.834     0.796     0.814       328

    accuracy                          0.795       581
   macro avg      0.792     0.795     0.793       581
weighted avg      0.797     0.795     0.796       581

Confusion Matrix:
 [[201  52]
 [ 67 261]]


In [25]:
!pip install faster-whisper opensmile transformers -q
!apt-get install -y ffmpeg -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 63.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.6/39.6 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 84.2 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 134 not upgraded.


In [26]:
!pip install openai-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 25.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803980 sha256=be9812dfd8ce652b299d085698e66805cf372fc7821eebbbafeb6094a2b70339
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper


In [27]:
# =============================================================================
# CELL 1 — SAVE THE TRAINED MulT MODEL + ARTIFACTS
# =============================================================================
import os, pickle
import torch

SAVE_DIR = "/kaggle/working/mult_saved"
os.makedirs(SAVE_DIR, exist_ok=True)

# 1. Model weights (using mult_model from Cell G)
torch.save(mult_model.state_dict(), f"{SAVE_DIR}/model_weights.pt")

# 2. vis_proj weights (ResNet 512 -> 35 projection layer)
torch.save(vis_proj.state_dict(), f"{SAVE_DIR}/vis_proj_weights.pt")

# 3. Config (MulT Hyperparameters)
config = {
    "TEXT_DIM":     TEXT_DIM,       # 768 (raw BERT, no PCA)
    "VISUAL_DIM":   VISUAL_DIM,     # 35
    "ACOUSTIC_DIM": ACOUSTIC_DIM,   # 74
    "PROJ_DIM":     PROJ_DIM,       # 64
    "NUM_HEADS":    NUM_HEADS,      # 2
    "NUM_CLASSES":  NUM_CLASSES,    # 2
    "DROPOUT":      DROPOUT,        # 0.4
    "MAX_UTT":      MAX_UTT,        # 63
}

with open(f"{SAVE_DIR}/config.pkl", "wb") as f:
    pickle.dump(config, f)

print("✓ Saved MulT artifacts to:", SAVE_DIR)
for fname in sorted(os.listdir(SAVE_DIR)):
    size = os.path.getsize(f"{SAVE_DIR}/{fname}") / 1024
    print(f"    {fname:35s}  {size:8.1f} KB")

✓ Saved MulT artifacts to: /kaggle/working/mult_saved
    config.pkl                                0.1 KB
    model_weights.pt                       2329.3 KB
    vis_proj_weights.pt                      72.1 KB


In [28]:
# =============================================================================
# CELL 2 — FULL VIDEO INFERENCE PIPELINE (MulT Architecture)
# =============================================================================
import os, pickle, re, warnings, subprocess, shutil, tempfile
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import cv2
import opensmile
import whisper
import torchvision.models as tvm
import torchvision.transforms as T
from transformers import AutoTokenizer, AutoModel

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =============================================================================
# STEP 0 - Load saved artifacts from MulT directory
# =============================================================================
SAVE_DIR = "/kaggle/working/mult_saved"

with open(f"{SAVE_DIR}/config.pkl", "rb") as f:
    cfg = pickle.load(f)

TEXT_DIM     = cfg["TEXT_DIM"]      # 768
VISUAL_DIM   = cfg["VISUAL_DIM"]    # 35
ACOUSTIC_DIM = cfg["ACOUSTIC_DIM"]  # 74
MAX_UTT      = cfg["MAX_UTT"]       # 63
PROJ_DIM     = cfg["PROJ_DIM"]      # 64
NUM_HEADS    = cfg["NUM_HEADS"]     # 2
NUM_CLASSES  = cfg["NUM_CLASSES"]   # 2
DROPOUT      = cfg["DROPOUT"]       # 0.4

# =============================================================================
# STEP 1 — Rebuild MulT model and load weights (FIXED)
# =============================================================================
model = MulT_PerUtterance(
    td=TEXT_DIM,       # 768
    vd=VISUAL_DIM,     # 35
    ad=ACOUSTIC_DIM,   # 74
    d=PROJ_DIM,        # 64
    nhead=NUM_HEADS,   # 2
    num_classes=NUM_CLASSES, # 2
    dropout=DROPOUT    # 0.4
).to(device)

model.load_state_dict(torch.load(f"{SAVE_DIR}/model_weights.pt", map_location=device))
model.eval()
print("✓ MulT model loaded successfully")

# =============================================================================
# STEP 2 - Initialise feature extractors
# =============================================================================

# Visual: ResNet18 -> vis_proj (512 -> 35)
resnet = tvm.resnet18(pretrained=True)
resnet = nn.Sequential(*list(resnet.children())[:-1]).to(device)
resnet.eval()

vis_proj = nn.Linear(512, VISUAL_DIM).to(device)
vis_proj.load_state_dict(torch.load(f"{SAVE_DIR}/vis_proj_weights.pt", map_location=device))
vis_proj.eval()

vis_transform = T.Compose([
    T.ToPILImage(),
    T.Resize(224), T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Text: BERT CLS -> raw 768-d
tokenizer  = AutoTokenizer.from_pretrained("bert-base-uncased")
bert_model = AutoModel.from_pretrained("bert-base-uncased").to(device)
bert_model.eval()

# Acoustic: openSMILE eGeMAPSv02 -> 74-d
smile = opensmile.Smile(
    feature_set=opensmile.FeatureSet.eGeMAPSv02,
    feature_level=opensmile.FeatureLevel.Functionals,
)

# ASR: Whisper small
whisper_model = whisper.load_model("small")

print("✓ All feature extractors ready")

# =============================================================================
# STEP 3 - ASR Transcribe & VAD Segmentation Functions
# =============================================================================

def transcribe_full_audio(video_path, tmp_dir):
    full_wav = os.path.join(tmp_dir, "_full_audio.wav")
    subprocess.run([
        "ffmpeg", "-y", "-i", video_path,
        "-ar", "16000", "-ac", "1",
        "-loglevel", "error", full_wav,
    ], check=True)

    print("  Whisper transcribing full audio...")
    result = whisper_model.transcribe(
        full_wav,
        fp16=torch.cuda.is_available(),
        word_timestamps=True,
        temperature=0.0,
        beam_size=5,
    )

    words = []
    for seg in result.get("segments", []):
        for w in seg.get("words", []):
            word_text = w["word"].strip()
            if word_text:
                words.append({"word": word_text, "start": w["start"], "end": w["end"]})

    print(f"  {len(words)} words transcribed")
    return words

def words_in_window(all_words, t_start, t_end, pad=0.15):
    chunk = [
        w["word"] for w in all_words
        if t_start - pad <= (w["start"] + w["end"]) / 2 <= t_end + pad
    ]
    return " ".join(chunk).strip()

def segment_video(video_path, out_dir, min_silence_db=-35, min_silence_dur=0.35, merge_gap=0.6, max_seg_dur=30.0, min_seg_dur=0.3):
    os.makedirs(out_dir, exist_ok=True)
    probe = subprocess.run([
        "ffmpeg", "-i", video_path,
        "-af", f"silencedetect=noise={min_silence_db}dB:d={min_silence_dur}",
        "-f", "null", "-",
    ], capture_output=True, text=True)
    stderr = probe.stderr

    silence_starts = [float(x) for x in re.findall(r"silence_start: ([\d.]+)", stderr)]
    silence_ends   = [float(x) for x in re.findall(r"silence_end: ([\d.]+)",   stderr)]

    m = re.search(r"Duration: (\d+):(\d+):([\d.]+)", stderr)
    if m:
        total_dur = int(m.group(1))*3600 + int(m.group(2))*60 + float(m.group(3))
    else:
        total_dur = 3600.0

    raw = []
    if not silence_starts:
        raw = [(0.0, total_dur)]
    else:
        prev = 0.0
        for i, ss in enumerate(silence_starts):
            if ss - prev >= min_seg_dur:
                raw.append((prev, ss))
            prev = silence_ends[i] if i < len(silence_ends) else ss + min_silence_dur
        if total_dur - prev >= min_seg_dur:
            raw.append((prev, total_dur))
    if not raw:
        raw = [(0.0, total_dur)]

    merged = [list(raw[0])]
    for (s, e) in raw[1:]:
        if s - merged[-1][1] <= merge_gap:
            merged[-1][1] = e
        else:
            merged.append([s, e])
    merged = [(s, e) for s, e in merged if e - s >= min_seg_dur]

    chunked = []
    for (s, e) in merged:
        dur = e - s
        if dur <= max_seg_dur:
            chunked.append((s, e))
        else:
            n = int(np.ceil(dur / max_seg_dur))
            cs = dur / n
            for k in range(n):
                chunked.append((round(s + k*cs, 3), round(min(s + (k+1)*cs, e), 3)))

    print(f"  VAD segments created: {len(chunked)}")

    clips = []
    for i, (t_start, t_end) in enumerate(chunked):
        dur     = t_end - t_start
        mp4_out = os.path.join(out_dir, f"seg_{i:03d}.mp4")
        wav_out = os.path.join(out_dir, f"seg_{i:03d}.wav")

        subprocess.run([
            "ffmpeg", "-y", "-ss", str(t_start), "-i", video_path,
            "-t", str(dur), "-c:v", "libx264", "-c:a", "aac",
            "-loglevel", "error", mp4_out,
        ], check=False)

        subprocess.run([
            "ffmpeg", "-y", "-ss", str(t_start), "-i", video_path,
            "-t", str(dur), "-ar", "16000", "-ac", "1",
            "-loglevel", "error", wav_out,
        ], check=False)

        clips.append({"index": i, "mp4_path": mp4_out, "wav_path": wav_out, "start": t_start, "end": t_end})
    return clips

# =============================================================================
# STEP 4 - Feature Extractors
# =============================================================================

def get_visual_feat(mp4_path, n_frames=5):
    try:
        cap   = cv2.VideoCapture(mp4_path)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total == 0:
            return np.zeros(VISUAL_DIM, dtype=np.float32)
        idxs = np.linspace(0, total - 1, min(n_frames, total), dtype=int)
        frames = []
        for idx in idxs:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
            ret, frame = cap.read()
            if ret:
                frames.append(vis_transform(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
        cap.release()
        if not frames:
            return np.zeros(VISUAL_DIM, dtype=np.float32)
        with torch.no_grad():
            batch    = torch.stack(frames).to(device)
            raw_feat = resnet(batch).squeeze(-1).squeeze(-1).mean(0, keepdim=True)
            feat     = vis_proj(raw_feat).squeeze(0)
        return feat.cpu().numpy().astype(np.float32)
    except Exception:
        return np.zeros(VISUAL_DIM, dtype=np.float32)

def get_acoustic_feat(wav_path):
    try:
        feat = smile.process_file(wav_path).values[0].astype(np.float32)
        if len(feat) >= ACOUSTIC_DIM:
            return feat[:ACOUSTIC_DIM]
        return np.pad(feat, (0, ACOUSTIC_DIM - len(feat))).astype(np.float32)
    except Exception:
        return np.zeros(ACOUSTIC_DIM, dtype=np.float32)

def get_text_feat(text):
    if not text:
        return np.zeros(TEXT_DIM, dtype=np.float32)
    try:
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=64, padding=True).to(device)
        with torch.no_grad():
            cls = bert_model(**inputs).last_hidden_state[:, 0, :]
        return cls.cpu().numpy()[0].astype(np.float32)
    except Exception:
        return np.zeros(TEXT_DIM, dtype=np.float32)

# =============================================================================
# STEP 5 - MAIN INFERENCE FUNCTION
# =============================================================================

def predict_video_sentiment(video_path, cleanup=True, verbose=True):
    print(f"\n{'='*65}\n INPUT : {os.path.basename(video_path)}\n{'='*65}")
    tmp_dir = tempfile.mkdtemp(prefix="mult_infer_")

    try:
        all_words = transcribe_full_audio(video_path, tmp_dir)
    except Exception as e:
        print(f"  Whisper failed: {e}")
        all_words = []

    clips = segment_video(video_path, tmp_dir)
    if not clips:
        print("  No segments detected.")
        return [], "Unknown"

    n_total = len(clips)
    print(f"\n  Extracting features for {n_total} segments...")

    txt_feats, vis_feats, aco_feats, transcripts = [], [], [], []
    for clip in clips:
        transcript = words_in_window(all_words, clip["start"], clip["end"]) if all_words else ""
        transcripts.append(transcript)
        txt_feats.append(get_text_feat(transcript))
        vis_feats.append(get_visual_feat(clip["mp4_path"]))
        aco_feats.append(get_acoustic_feat(clip["wav_path"]))

    all_probs = np.zeros((n_total, 2), dtype=np.float32)
    counts    = np.zeros(n_total, dtype=np.int32)

    stride = MAX_UTT // 2
    starts = list(range(0, n_total, stride))
    if n_total > MAX_UTT and starts[-1] + MAX_UTT < n_total:
        starts.append(n_total - MAX_UTT)

    def pad_seq(lst, dim):
        arr = np.stack(lst)
        if arr.shape[0] < MAX_UTT:
            arr = np.concatenate([
                arr,
                np.zeros((MAX_UTT - arr.shape[0], dim), dtype=np.float32),
            ], axis=0)
        return arr

    for w_start in starts:
        w_end  = min(w_start + MAX_UTT, n_total)
        w_size = w_end - w_start
        with torch.no_grad():
            t = torch.tensor(pad_seq(txt_feats[w_start:w_end], TEXT_DIM), dtype=torch.float32).unsqueeze(0).to(device)
            v = torch.tensor(pad_seq(vis_feats[w_start:w_end], VISUAL_DIM), dtype=torch.float32).unsqueeze(0).to(device)
            a = torch.tensor(pad_seq(aco_feats[w_start:w_end], ACOUSTIC_DIM), dtype=torch.float32).unsqueeze(0).to(device)
            
            # Forward pass through MulT
            logits = model(t, v, a) # Expected shape: (1, MAX_UTT, 2)
            probs = F.softmax(logits, dim=-1)[0].cpu().numpy()

        for j in range(w_size):
            all_probs[w_start + j] += probs[j]
            counts[w_start + j]    += 1

    all_probs /= np.maximum(counts, 1)[:, None]
    preds      = all_probs.argmax(axis=1)
    labels     = ["Negative", "Positive"]

    results = []
    for i in range(n_total):
        results.append({
            "segment":    i + 1,
            "start_sec":  round(clips[i]["start"], 2),
            "end_sec":    round(clips[i]["end"],   2),
            "transcript": transcripts[i],
            "prediction": labels[preds[i]],
            "confidence": round(float(all_probs[i].max()) * 100, 1),
            "neg_prob":   round(float(all_probs[i][0]) * 100, 1),
            "pos_prob":   round(float(all_probs[i][1]) * 100, 1),
        })

    pos_score    = sum(r["pos_prob"] for r in results)
    neg_score    = sum(r["neg_prob"] for r in results)
    overall      = "Positive" if pos_score >= neg_score else "Negative"
    overall_conf = round(max(pos_score, neg_score) / (pos_score + neg_score) * 100, 1)

    print(f"\n{'─'*65}\n  {'SEG':>4}  {'TIME':>14}  {'PRED':>10}  {'CONF':>7}  TRANSCRIPT\n{'─'*65}")
    for r in results:
        time_str = f"{r['start_sec']}s-{r['end_sec']}s"
        print(f"  {r['segment']:>4}  {time_str:>14}  {r['prediction']:>10}  "
              f"{r['confidence']:>6.1f}%  {r['transcript'][:42]}")
    print(f"{'─'*65}")
    print(f"  OVERALL  ->  {overall}  ({overall_conf:.1f}% confidence)")
    print(f"  Segments ->  {n_total}\n{'='*65}\n")

    if cleanup:
        shutil.rmtree(tmp_dir, ignore_errors=True)

    return results, overall

✓ MulT model loaded successfully


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
100%|████████████████████████████████████████| 461M/461M [00:03<00:00, 158MiB/s]


✓ All feature extractors ready


In [29]:
VIDEO_PATH = "/kaggle/input/datasets/koushikshaw2916/videotest/0h-zjBukYpk.mp4"   # <- change this
 
results, overall = predict_video_sentiment(VIDEO_PATH, cleanup=True, verbose=True)


 INPUT : 0h-zjBukYpk.mp4
  Whisper transcribing full audio...
  400 words transcribed
  VAD segments created: 32

  Extracting features for 32 segments...

─────────────────────────────────────────────────────────────────
   SEG            TIME        PRED     CONF  TRANSCRIPT
─────────────────────────────────────────────────────────────────
     1     3.75s-4.32s    Positive    55.5%  
     2      6.7s-7.66s    Positive    60.7%  
     3   10.97s-15.46s    Positive    93.9%  this is KeckmanEleven here for a review of
     4    17.12s-27.6s    Positive    53.2%  Hugh Jackman, Leaf Shrieber, Danny Houston
     5   28.45s-33.65s    Negative    87.7%  most specifically July 27th, so who wrong 
     6    35.38s-43.8s    Positive    82.3%  Now, the title of the movie basically says
     7    44.99s-46.9s    Negative    95.1%  Now, I'm not even going to sugarcoat this.
     8   47.52s-56.65s    Negative    90.5%  This movie frustrated me to such an extrem
     9   57.28s-63.67s    Negative 

In [30]:
VIDEO_PATH = "/kaggle/input/datasets/koushikshaw2916/videotest/test4.mp4"   # <- change this
 
results, overall = predict_video_sentiment(VIDEO_PATH, cleanup=True, verbose=True)


 INPUT : test4.mp4
  Whisper transcribing full audio...
  303 words transcribed
  VAD segments created: 4

  Extracting features for 4 segments...

─────────────────────────────────────────────────────────────────
   SEG            TIME        PRED     CONF  TRANSCRIPT
─────────────────────────────────────────────────────────────────
     1     0.0s-26.05s    Negative    90.3%  A question I'm often asked is, what is the
     2   26.05s-52.11s    Negative    69.2%  burned in my psyche, like some kind of fev
     3   52.75s-69.22s    Negative    59.0%  something like that. Maybe that made sense
     4    69.22s-85.7s    Positive    75.4%  and Let's Be Evil. And that night on strea
─────────────────────────────────────────────────────────────────
  OVERALL  ->  Negative  (60.8% confidence)
  Segments ->  4

